# 01 · EDA — PRC Data Challenge 2026 (taxi-out time)

**Objetivo:** entender los datos antes de modelar `TAXITIME_SEC_mvt` para movimientos `PHASE_mvt == "DEP"`.

**Reglas que respeta este notebook**
- Solo lee los 14 Parquet originales (nunca los escribe).
- Todo el cálculo pesado está en `src/eda.py` (Polars, proyección de columnas); el notebook solo orquesta y muestra.
- No se eliminan outliers: se cuantifican y se documentan.
- Tablas → `outputs/tables/*.csv`; figuras → `outputs/figures/*.png`.

Ejecutar de principio a fin: `jupyter nbconvert --to html --execute notebooks/01_eda.ipynb --output-dir outputs/reports --output eda_report.html`

In [ ]:
import os, sys
from pathlib import Path
os.environ["POLARS_UNKNOWN_EXTENSION_TYPE_BEHAVIOR"] = "load_as_storage"   # los Parquet vienen de R
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display
import eda
from config import DATA_DIR, FIGURES_DIR, SEED, TARGET

pl.Config.set_tbl_rows(45); pl.Config.set_tbl_cols(20); pl.Config.set_tbl_width_chars(220); pl.Config.set_fmt_str_lengths(70)
show = lambda name: display(Image(filename=str(FIGURES_DIR / f"{name}.png")))
print("Datos:", DATA_DIR, "| seed:", SEED)

## 1. Diccionario de las 30 variables

Clasificación: **identificadores**, **categóricas**, **temporales**, **numérica (target)**. Se muestra % de nulos y cardinalidad
en training-DEP y en ranking-DEP, y si la variable está oculta en ranking (→ no utilizable).

In [ ]:
dic = eda.data_dictionary()
dic

**Lectura.** Las columnas `_flt` (plan de vuelo) faltan en bloque (~1.1 % train / ~1.5 % ranking): son movimientos sin
enlace a plan de vuelo. `BLOCK_TIME_UTC_mvt` y el target están 100 % ocultos en ranking-DEP.

## 2. Valores más frecuentes por categórica (training-DEP)

In [ ]:
tv = eda.top_values(k=5)
tv

## 3. Carga de DEP (training y ranking) con variables de calendario
Se usan solo las columnas necesarias (`eda.BASE_COLS`). Las cadenas `"NA"` de `RUNWAY_mvt` se convierten a nulo en memoria.

In [ ]:
tr, rk = eda.load_dep()
print(f"training DEP: {tr.height:,} filas · ranking DEP: {rk.height:,} filas")
print("training:", tr['MVT_TIME_UTC_mvt'].min(), "→", tr['MVT_TIME_UTC_mvt'].max())
print("ranking :", rk['MVT_TIME_UTC_mvt'].min(), "→", rk['MVT_TIME_UTC_mvt'].max())
rk.group_by("month").len().sort("month")

> ⚠️ **El ranking NO cubre enero–julio de forma continua: solo contiene enero 2026 y julio 2026.** Esto condiciona el diseño del
backtesting: hay que validar en un mes de invierno y en uno de verano, y con horizonte de 1–7 meses respecto al final del training.

## 4. Distribución del target (solo DEP)

In [ ]:
eda.target_distribution(tr)

In [ ]:
show("target_hist_dep")

## 5. Negativos, ceros y valores extremos
No se elimina nada. Se cuantifica con varios umbrales para decidir después el tratamiento (recorte, winsorización, pérdida robusta…).

In [ ]:
eda.outliers(tr)

In [ ]:
display(pl.read_csv(FIGURES_DIR.parent / "tables" / "outliers_gt2h_by_airport.csv"))
display(pl.read_csv(FIGURES_DIR.parent / "tables" / "outliers_negative_by_airport.csv"))
show("target_daily_dep")

In [ ]:
# ¿Los extremos se concentran en movimientos sin plan de vuelo?
tr.with_columns(pl.col("AOBT_3_flt").is_null().alias("sin_plan_de_vuelo")).group_by("sin_plan_de_vuelo").agg(
    pl.len().alias("n"), pl.col(TARGET).median().alias("mediana"), pl.col(TARGET).std().round(0).alias("std"),
    (pl.col(TARGET) > 3600).mean().round(4).alias("pct_>1h"), (pl.col(TARGET) < 0).mean().round(5).alias("pct_neg"))

**Lectura.** Los negativos son minúsculos (−1 a −12 s: jitter de sensores, casi todos en LSZH y LFPG) y no representan errores
graves; pueden recortarse a 0. Los extremos (> 2 h) se concentran en **LIRF** (480 de 584) y en movimientos **sin plan de vuelo**
(4.8 % > 1 h frente a 0.15 %). Esto sugiere errores de registro de `BLOCK_TIME` más que taxi-out reales.

## 6. Taxi-out por aeropuerto, mes, día de semana y hora

In [ ]:
eda.by_airport(tr)

In [ ]:
show("taxi_by_airport_top")

In [ ]:
cal = eda.by_calendar(tr)
display(cal["month"]); display(cal["dow"]); display(cal["is_weekend"])
show("taxi_by_calendar"); show("departures_by_hour")

**Lectura.** Solo hay **10 aeropuertos de salida** y el efecto aeropuerto es enorme (mediana 711 s en LSZH vs 1 319 s en EGLL).
La estacionalidad mensual es débil (medianas 904–948 s); la hora del día importa más (picos de mañana y noche). El fin de semana
apenas cambia la mediana.

## 7. Taxi-out por pista, stand, aeronave, operador, segmento de mercado y estela

In [ ]:
op = eda.by_operational(tr)
display(op["runway"].head(20)); display(op["stand"].head(20))

In [ ]:
display(op["aircraft"].head(20)); display(op["operator"].head(15)); display(op["segment"]); display(op["wtc"])
show("taxi_by_segment_aircraft")

### ¿Cuánta varianza explica cada corte? (in-sample, cota optimista)
R² de sustituir cada fila por la media de su grupo y MAE con la mediana del grupo. Sirve para ordenar la utilidad de las variables
y como referencia para las baselines de la fase 4.

In [ ]:
op["explained"]

## 8. Training (2025) vs ranking (2026)
¿Cambia la mezcla de aeropuertos, horas, categorías? ¿Aparecen categorías nuevas? ¿Hay deriva en el taxi-IN (que sí es observable en ranking)?

In [ ]:
cmp = eda.train_vs_ranking(tr, rk)
display(cmp["coverage"]); display(cmp["airport_share"].head(10)); display(cmp["flt_nulls"])
show("train_vs_rank_airport_hour")

In [ ]:
display(cmp["taxi_in_drift"])
show("taxi_in_drift")

**Lectura.** Mezcla de aeropuertos y horas casi idéntica; < 0.5 % de filas del ranking con stand no visto y ~1.6 % con operador
no visto (hay que prever categorías desconocidas en el encoding). El taxi-in mediano por aeropuerto 2026 difiere de 2025 en ≤ 20 s:
**no hay señales de deriva estructural**, lo que hace razonable extrapolar del 2025 al 2026.

## 9. Redundancia y riesgo de *data leakage*

Se evalúan las diferencias temporales que **sí** pueden calcularse en ranking-DEP (todas las marcas menos `BLOCK_TIME`) como
predictores directos del target, y el grado de redundancia entre columnas `_mvt` / `_flt`.

In [ ]:
lk = eda.leakage_and_redundancy(tr, rk)
display(lk["deltas"]); display(lk["redundancy"]); display(lk["resolution"])
show("baseline_aobt3_error")

**Resolución de las marcas de tiempo** (fracción con `segundos == 0`): 0.017 ≈ 1 s, 0.083 ≈ 5 s, ≈ 1.0 = 1 min.
`AOBT_3_flt` y `EOBT_1_flt` están a minuto; `BLOCK_TIME`/`MVT_TIME` a 1 s (EDDF, EHAM, LSZH…) o 5 s (EGLL, LFPG, LIRF, LTFM, EDDM).
El error de `MVT_TIME − AOBT_3` tiene por tanto un piso de ±30 s por redondeo, además del desfase real entre sistemas.

In [ ]:
lk["risk"]

**Lectura.**
- `TAXITIME_SEC_mvt == MVT_TIME − BLOCK_TIME` exactamente → `BLOCK_TIME` es el target disfrazado y está (correctamente) oculto.
- `MVT_TIME − AOBT_3_flt` es la **baseline operacional**: disponible en el 98.9 % de las filas, pero **no** es casi-perfecta:
  MAE ≈ 238 s, RMSE ≈ 385 s y solo el 21 % de errores ≤ 60 s. `AOBT_3` (reporte A-CDM/DPI) difiere del `BLOCK_TIME` del aeropuerto
  con mediana +51 s y colas anchas (p5 = −513 s, p95 = +512 s), y su calidad varía mucho por aeropuerto (LTFM: mediana +296 s).
  → Hay margen claro para un modelo de **residuo** sobre esa baseline.
- `MVT_TIME − EOBT_1` (planificado) es el mejor *fallback* cuando no hay `AOBT_3` (misma disponibilidad, así que en la práctica
  el fallback tendrá que ser la mediana por grupo).
- `ADEP_flt`, `AIRCRAFT_TYPE_flt`, `FLIGHT_RULE_flt` son redundantes (> 99.6 %) con sus homólogas `_mvt`; `ADES_FILED` ≈ `ADES`.
- `ARVT_1/ARVT_3` son posteriores al despegue: legales (están en ranking) pero solo informan de la duración del vuelo.

## 10. Señal de congestión: salidas por hora en el mismo aeropuerto

In [ ]:
eda.congestion_signal(tr)

In [ ]:
show("congestion_vs_taxi")

**Lectura.** En EGLL, LEBL, LTFM, EDDF y LFPG el nº de salidas por hora correlaciona positivamente con el taxi-out mediano
(ρ 0.25–0.50): las variables de **congestión** (salidas/llegadas en ventanas de ±15/30/60 min, calculables en ranking con
`MVT_TIME` de todas las filas, ARR incluidas) son candidatas prioritarias para la fase 3.

## 11. Conclusiones para las fases 3–5

1. **Target**: mediana 912 s, media 991 s, cola larga (p99 = 2 339 s; máximo 131 167 s). Negativos irrelevantes (≥ −12 s). Extremos
   concentrados en LIRF y en movimientos sin plan de vuelo → evaluar recorte del target en training (p. ej. a 2 h o p99.9) **sin
   tocar** las filas de ranking.
2. **Variables más prometedoras**: `ADEP_mvt` (y sus interacciones), `STAND_mvt`, `RUNWAY_mvt`, `AIRCRAFT_TYPE`/`WK_TBL_CAT`,
   hora del día, `MVT_TIME − AOBT_3` y `MVT_TIME − EOBT_1`, congestión por ventana temporal, operador y segmento.
3. **Leakage**: `BLOCK_TIME` prohibida; `AOBT_3` permitida pero domina; ids no se usan como predictores; el encoding de
   categóricas debe ajustarse solo con el bloque de entrenamiento de cada backtest.
4. **Validación temporal**: el ranking contiene solo enero 2026 y julio 2026 → además de los tres backtests pedidos
   (→ jul, → nov, → dic), es prudente mirar el rendimiento con horizontes de 1 y 7 meses.
5. **Primer baseline recomendado**: mediana por `ADEP × RUNWAY × hora × tipo de aeronave` con *back-off* jerárquico
   (→ `ADEP × RUNWAY` → `ADEP`) y, en paralelo, la baseline operacional `MVT_TIME − AOBT_3`; el primer modelo será un CatBoost
   sobre el **residuo** de la baseline operacional con fallback a la mediana jerárquica.